<a href="https://colab.research.google.com/github/AtulThakur-hub-glitch/Model_Comparison.py/blob/main/NLP_Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#1.Import Libraries
import pandas as pd
import numpy as np
import re
import string
import matplotlib.pyplot as plt

#NLTK
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

#spaCy
import spacy

#Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import  TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, classification_report, precision_score, recall_score, f1_score,confusion_matrix,ConfusionMatrixDisplay)

#2.Download NLTK Resources
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

#3.Load Dataset
nlp=spacy.load('en_core_web_sm')

#4.Load Dataset
file_path = "twitter_training.csv"
df=pd.read_csv(file_path,header=None)

#5.Give Meaningful Column Names
df.columns = ["Tweet_ID", "Entity","Sentiment", "Text"]

print("First five records:")
print(df.head())

#6. Basic Dataset Information
print("\nDataset Shape:", df.shape)
print("\nDataset Information:",df.info())
print("\nMissing Values:\n",df.isnull().sum())
print("\nSentiment Distribution:")
print(df['Sentiment'].value_counts())

#7. Remove Missing Values
df=df.dropna(subset=["Text","Sentiment"])
print("\nDataset Shape after removing missing values:", df.shape)

#8.Remove Irrelevant Class
df=df[df['Sentiment'].isin(['Positive','Negative','Neutral'])]

print("\nSentiment Distribution after removing Irrelevant:")
print(df['Sentiment'].value_counts())

#9. Text Preprocessing Function
stop_words=set(stopwords.words('english'))
lemmatizer=WordNetLemmatizer()
def clean_text(text):
    text=text.lower()
    text=re.sub(r"http\S+","", text)
    text=re.sub(r'@\W+','', text)
    text=re.sub(r'#','', text)
    text=re.sub(r'\d+','', text)
    text=text.translate(str.maketrans('', '', string.punctuation))
    text=re.sub(r'\s+',' ', text).strip()
    words=text.split()
    words=[word for word in words if word not in stop_words]
    return ' '.join(words)
df["clean_text"]=df["Text"].apply(clean_text)
print(df.head())
df=df[df["clean_text"].str.strip()!=""]

#12.Optional Spacy Processing
def spacy_preprocess(text):
    doc=nlp(text)
    tokens=[]
    for token in doc:
        if(
            not token.is_stop
            and not token.is_punct
            and not token.is_space
        ):
            tokens.append(token.lemma_.lower())
    return "".join(tokens)

df["Processed_Text"]=df["clean_text"].apply(spacy_preprocess)

print("\nText after spaCy preprocessing:")
print(df[["Text","Processed_Text"]].head(10))

#13.Define features and target
X=df["Processed_Text"]
y=df["Sentiment"]

#14.Split data into training and testing sets
X_train,X_test,y_train,y_test=train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
print("\nTraining samples:",len(X_train))
print("Testing samples:",len(X_test))

#15.TF-IDF feature extraction
vectorizer=TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95

)

X_train_tfidf=vectorizer.fit_transform(X_train)
X_test_tfidf=vectorizer.transform(X_test)

print("\nTF-IDF Training Shape:")
print(X_train_tfidf.shape)

print("\nTF-IDF Testing Shape:")
print(X_test_tfidf.shape)

# 17. Train The Model
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

# 18. Make Predictions
y_pred = model.predict(X_test_tfidf)

#19. Calculate Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("\nAccuracy:", accuracy)
print(f"{accuracy:.4f}")

# 20. Calculate Precision
precision = precision_score(y_test, y_pred, average='weighted')
print("\nPrecision:", precision)
print(f"{precision:.4f}")

# 21. Calculate Recall
recall = recall_score(y_test, y_pred, average='weighted')
print("\nRecall:", recall)
print(f"{recall:.4f}")

#22. Calculate F1-Score
f1 = f1_score(y_test, y_pred, average='weighted')
print("\nF1-Score:", f1)
print(f"{f1:.4f}")

#23. Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

#24. Confusion Matrix
cm = confusion_matrix(y_test, y_pred,labels=['Positive', 'Negative', 'Neutral'])
print("\nConfusion Matrix:")
print(cm)

#25. Confusion Matrix Visualization
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Positive', 'Negative', 'Neutral'])
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix - Naive Bayes")
plt.show()

#
df["Sentiment"].value_counts().plot(kind='bar')
plt.title("Sentiment Distribution")
plt.xlabel('Sentiment')
plt.ylabel('No. of Tweets')
plt.xticks(rotation=0)
plt.show()

#Function To Predict New Text
def predict_sentiment(text):
  #Clean the new text using the same preprocessing
    cleaned=clean_text(text)
    #Apply spaCy preprocessing
    text=spacy_preprocess(cleaned)
    #Convertion
    text_vectorized=vectorizer.transform([text]) # Changed 'processed' to 'text'
  #Predict Sentiment
    prediction=model.predict(text_vectorized)[0]
    #Get prediction probabilities
    probabilities  = model.predict_proba(text_vectorized)[0] # Corrected 'probabllities' and 'text_vector'

#Display Result
    print("\nInput Text:")
    print(text)

    print("\nPredicted Sentiment:")
    print(prediction)

    print("\nPrediction Probabilities:")
    for sentiment, probability in zip(model.classes_, probabilities):
        print(f"{sentiment}: {probability:.4f}")

        return prediction